# NUBI Video AI Studio · Google Colab
Publique os arquivos do pacote na raiz de **FelipeGomes070898/NUBI-ORCHESTRATOR** e escolha **Ambiente de execução → Executar tudo**.

**Duas formas de usar:** (A) o **Dashboard** da seção 3, que abre dentro do Colab; (B) as células de configuração e renderização manual (seções 4 a 6). Estilos: `cartoon` (2D com contornos e mascote), `parallax` (camadas em velocidades diferentes), `3d` (2.5D com câmera em perspectiva) e `zoom` (modo original).

## 1. Baixar o projeto do GitHub

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORIO = "https://github.com/FelipeGomes070898/NUBI-ORCHESTRATOR.git"
PROJETO = Path("/content/NUBI-ORCHESTRATOR")
if not PROJETO.exists():
    try:
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORIO, str(PROJETO)],
                       check=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    except subprocess.CalledProcessError as erro:
        raise RuntimeError("Confira se o repositório está público e contém os arquivos extraídos do ZIP.") from erro
if not all((PROJETO / nome).is_file() for nome in ["nubi/render.py", "requirements.txt"]):
    raise FileNotFoundError("A raiz deve conter nubi/render.py e requirements.txt. Confira a estrutura no GitHub. Se o código foi publicado depois da clonagem, inicie uma sessão nova.")
os.chdir(PROJETO)
print("Versão em execução:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

## 2. Instalar os motores de vídeo e voz

In [ ]:
import shutil
if any(shutil.which(cmd) is None for cmd in ("ffmpeg", "ffprobe", "espeak-ng")):
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "espeak-ng"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
for cmd in ("ffmpeg", "ffprobe", "espeak-ng"):
    print(cmd, shutil.which(cmd))

## 3. Abrir o Dashboard (recomendado)
Inicia o servidor local e o exibe abaixo, com o fluxo ideia → diretor → camadas/narração → composição → vídeo final. Marque SALVAR_NO_DRIVE_PAINEL para manter os vídeos no Google Drive.

In [ ]:
SALVAR_NO_DRIVE_PAINEL = False # @param {type:"boolean"}
from nubi.dashboard import serve
from google.colab import output
destino_painel = "/content/NUBI-producoes"
if SALVAR_NO_DRIVE_PAINEL:
    from google.colab import drive
    drive.mount("/content/drive")
    destino_painel = "/content/drive/MyDrive/NUBI-producoes"
serve(8765, destino_painel)
output.serve_kernel_port_as_iframe(8765, height=1100)


## 4. Configurar o vídeo
Para narrar em português, escolha **pt-br** e escreva o roteiro em português entre as três aspas, uma cena por linha. Para usar fotos próprias, carregue-as no painel **Arquivos** e preencha IMAGENS com um caminho por cena, na ordem do roteiro. Sem fotos, o programa cria ilustrações abstratas de tema espacial. Com **SALVAR_NO_DRIVE** ativado, os arquivos das cenas ficam disponíveis para retomar o projeto em outra sessão.

In [ ]:
TITULO = "How the Universe Began" # @param {type:"string"}
IDIOMA = "en" # @param ["en", "pt-br"]
DURACAO = 60 # @param {type:"integer"}
ESTILO = "cartoon" # @param ["zoom", "parallax", "cartoon", "3d"]
FORMATO = "16:9" # @param ["16:9", "9:16"]
RESOLUCAO = 1080 # @param [360, 720, 1080] {type:"raw"}
SALVAR_NO_DRIVE = False # @param {type:"boolean"}

ROTEIRO = """
"""
IMAGENS = []  # Exemplo: ["/content/foto1.jpg", "/content/foto2.jpg"]

DESTINO = Path("/content/NUBI-producoes")
if SALVAR_NO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DESTINO = Path("/content/drive/MyDrive/NUBI-producoes")

## 5. Renderizar e verificar o MP4
A saída identifica a cena em processamento. Se a sessão for interrompida, execute novamente com os mesmos parâmetros e a mesma pasta de destino para aproveitar as cenas concluídas.

In [ ]:
from nubi.render import render, probe
from IPython.display import Video, display

pasta = render(TITULO, ROTEIRO, IDIOMA, DURACAO, FORMATO == "9:16", RESOLUCAO, DESTINO, IMAGENS, ESTILO)
video = pasta / "video.mp4"
dados = probe(video)
print(f"Arquivo verificado: {video.name} · {float(dados['format']['duration']):.2f} s")
display(Video(str(video), embed=True, width=480))

## 6. Baixar o resultado
Escolha MP4 para levar o vídeo ou ZIP para incluir as imagens, vozes, cenas, miniatura, legendas, metadados e log.

In [ ]:
from google.colab import files
BAIXAR = "MP4" # @param ["MP4", "ZIP"]
arquivo = video
if BAIXAR == "ZIP":
    arquivo = shutil.make_archive(str(Path("/content") / ("NUBI-" + pasta.name)), "zip", pasta)
files.download(str(arquivo))